# Preparing - Load library `aleph`

In [1]:
!pip install -q torch_geometric
!pip install -q build
!pip install scanpy
!python -m build --wheel

%cp -r /kaggle/input/aleph-package/Aleph/ /kaggle/working/Aleph/
%cd /kaggle/working/Aleph/build/bindings/python/aleph
!python setup.py install
!pip install .

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 31.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 111.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 110.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 118.2 MB/s eta 0:00:0000:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2

## Load library

In [2]:
import os
import sys
import h5py
import numpy as np
import torch
import random
from torch_geometric.data import Data
from scipy.spatial.distance import pdist, squareform
from sklearn.neighbors import NearestNeighbors
import networkx as nx

# 1. Module Code

## 1.1 Preprocessing

In [3]:
import scanpy as sc
from sklearn.preprocessing import LabelEncoder

def normalization(features_):
    features = features_.copy()
    for i in range(len(features)):
        features[i] = features[i] / sum(features[i]) * 100000
    features = np.log2(features + 1)
    return features

def preprocess(X, nb_genes = 2000):
    """
    Preprocessing phase as proposed in scanpy package. Keeps only nb_genes most variable genes and normalizes the data to 0 mean and 1 std.
    """

    print("There are " + str(X.shape[0]) + " cells, " + str(X.shape[1]) + " genes")
    print("Data Preprocessing!")
    
    adata = sc.AnnData(X)
       
    adata = normalize_1(adata,
                      copy=True,
                      highly_genes=nb_genes,
                      size_factors=True,
                      normalize_input=True,
                      logtrans_input=True)
    X = adata.X.astype('float32')
    print(f"Keeping {nb_genes} genes")
    return X

def load_h5_data1(data_path):
    print("Reading data!")
    inputFile = h5py.File(data_path, 'r')
   
    X = np.array(inputFile['X']).astype('float32')
    Y = np.array(inputFile['Y'])

    if Y.dtype != "int64":
        encoder_x = LabelEncoder()
        Y = encoder_x.fit_transform(Y)
    inputFile.close()
    
    X=preprocess(X, nb_genes=2000)
    return X, Y

def normalize_1(adata, copy=True, highly_genes = None, filter_min_counts=True, 
              size_factors=True, normalize_input=True, logtrans_input=True):
    """
    Normalizes input data and retains only most variable genes 
    (indicated by highly_genes parameter)

    Args:
        adata ([type]): [description]
        copy (bool, optional): [description]. Defaults to True.
        highly_genes ([type], optional): [description]. Defaults to None.
        filter_min_counts (bool, optional): [description]. Defaults to True.
        size_factors (bool, optional): [description]. Defaults to True.
        normalize_input (bool, optional): [description]. Defaults to True.
        logtrans_input (bool, optional): [description]. Defaults to True.

    Raises:
        NotImplementedError: [description]

    Returns:
        [type]: [description]
    """
    if isinstance(adata, sc.AnnData):
        if copy:
            adata = adata.copy()
    elif isinstance(adata, str):
        adata = sc.read(adata)
    else:
        raise NotImplementedError
    norm_error = 'Make sure that the dataset (adata.X) contains unnormalized count data.'
    assert 'n_count' not in adata.obs, norm_error
    

    if filter_min_counts:
        sc.pp.filter_genes(adata, min_cells=1)
        #sc.pp.filter_cells(adata, min_genes=1)
    if size_factors or normalize_input or logtrans_input:
        adata.raw = adata.copy()
    else:
        adata.raw = adata
    if size_factors:
        sc.pp.normalize_total(adata, exclude_highly_expressed=True)

    if logtrans_input:
        sc.pp.log1p(adata)
    if highly_genes != None:
        sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, n_top_genes = highly_genes, subset=True)

    return adata

def GraphConstruction(data, features, num_clusters):
    cell_num = features.shape[0]
    average_num = cell_num // num_clusters
    neighbor_num = average_num // 10
    neighbor_num = min(neighbor_num, 15)
    neighbor_num = max(neighbor_num, 5)
 
    # Calculate Pearson distance matrix
    dis_matrix = squareform(pdist(features, metric='correlation'))
    # Build kNN graph
    nbrs = NearestNeighbors(n_neighbors=neighbor_num, metric='precomputed').fit(dis_matrix)
    _, indices = nbrs.kneighbors(dis_matrix)  # Get only indices
 
    # Create adjacency matrix
    n_samples = features.shape[0]
    adj_matrix = np.zeros((n_samples, n_samples))

    for i in range(n_samples):
        for j in indices[i]:
             adj_matrix[i, j] = 1  

      
    #Create a list of egdes
    edge_list = torch.empty((2,0), dtype=torch.int64)
    for i in range( adj_matrix.shape[0]):
     for j in range(i + 1,   adj_matrix.shape[1]):
        if  adj_matrix[i, j] == 1:
            col = torch.tensor([i, j], dtype=torch.int64)
            edge_list = torch.cat((edge_list, col.unsqueeze(1)), dim=1)
    data.edge_index=edge_list
          
    #generate a graph
    G = nx.from_numpy_array(adj_matrix)
    print("Building a " + str(G))
    print("===================================================")

    return G

## 1.2 Model Architecture

In [5]:
from torch import nn
from torch.nn import Module, Parameter
import math
import torch.nn.functional as F

class GCNConv(Module):
    def __init__(self, in_features, out_features, bias=True):
        super(GCNConv, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = Parameter(torch.FloatTensor(in_features, out_features))
        if bias:
            self.bias = Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1. / math.sqrt(self.weight.size(1))
        self.weight.data.uniform_(-stdv, stdv)
        if self.bias is not None:
            self.bias.data.uniform_(-stdv, stdv)

    def forward(self, input, adj):
        support = torch.mm(input, self.weight)
        output = torch.spmm(adj, support)
        if self.bias is not None:
            return output + self.bias
        else:
            return output

    def __repr__(self):
        return self.__class__.__name__ + ' (' \
               + str(self.in_features) + ' -> ' \
               + str(self.out_features) + ')'

class Encoder(nn.Module):
    def __init__(self, dim_in: int, dim_out: int, num_layers: int = 2):
        super(Encoder, self).__init__()
        self.base_model = GCNConv
        self.activation = F.relu
        assert num_layers >= 2
        self.num_layers= num_layers
        self.conv = [GCNConv(dim_in, 2 * dim_out)]
        for _ in range(1, num_layers-1):
            self.conv.append(GCNConv(2 * dim_out, 2 * dim_out))
        self.conv.append(GCNConv(2 * dim_out, dim_out))
        self.conv = nn.ModuleList(self.conv)


    def forward(self, x: torch.Tensor, edge_index: torch.Tensor):
        for i in range(self.num_layers):
            x = self.activation(self.conv[i](x, edge_index))
        return x
    
class AGCLModel(torch.nn.Module):
    def __init__(self, encoder: Encoder, n_hidden: int, n_proj_hidden: int,
                 tau: float = 0.5):
        super(AGCLModel, self).__init__()
        self.encoder: Encoder = encoder
        self.tau: float = tau

        self.fc_layer1 = torch.nn.Linear(n_hidden, n_proj_hidden)
        self.fc_layer2 = torch.nn.Linear(n_proj_hidden, n_hidden)
      
        
    def forward(self, x: torch.Tensor,
                Adj: torch.Tensor) -> torch.Tensor:
       
        return self.encoder(x, Adj)

    def projection(self, z: torch.Tensor) -> torch.Tensor:
        z = F.elu(self.fc_layer1(z))
        return self.fc_layer2(z)

    #Codes are modified from https://github.com/Shengyu-Feng/ARIEL
    def sim(self, z1: torch.Tensor, z2: torch.Tensor):
        z1 = F.normalize(z1)
        z2 = F.normalize(z2)
        return torch.mm(z1, z2.t())

    def semi_loss(self, z1: torch.Tensor, z2: torch.Tensor):
        f = lambda x: torch.exp(x / self.tau)
        refl_sim = f(self.sim(z1, z1))
        between_sim = f(self.sim(z1, z2))

        return -torch.log(
            between_sim.diag()
            / (refl_sim.sum(1) + between_sim.sum(1) - refl_sim.diag()))

    def batched_semi_loss(self, z1: torch.Tensor, z2: torch.Tensor,
                          batch_size: int):
        # Space complexity: O(BN) (semi_loss: O(N^2))
        device = z1.device
        num_nodes = z1.size(0)
        num_batches = (num_nodes - 1) // batch_size + 1
        f = lambda x: torch.exp(x / self.tau)
        indices = torch.arange(0, num_nodes).to(device)
        losses = []

        for i in range(num_batches):
            mask = indices[i * batch_size:(i + 1) * batch_size]
            refl_sim = f(self.sim(z1[mask], z1))  # [B, N]
            between_sim = f(self.sim(z1[mask], z2))  # [B, N]

            losses.append(-torch.log(
                between_sim[:, i * batch_size:(i + 1) * batch_size].diag()
                / (refl_sim.sum(1) + between_sim.sum(1)
                   - refl_sim[:, i * batch_size:(i + 1) * batch_size].diag())))

        return torch.cat(losses)

    def loss(self, z1: torch.Tensor, z2: torch.Tensor,
             mean: bool = True, batch_size: int = 0):
        h1 = self.projection(z1)
        h2 = self.projection(z2)
        
            
        if batch_size == 0:
            l1 = self.semi_loss(h1, h2)
            l2 = self.semi_loss(h2, h1)
        else:
            l1 = self.batched_semi_loss(h1, h2, batch_size)
            l2 = self.batched_semi_loss(h2, h1, batch_size)

        ret = (l1 + l2) * 0.5
        ret = ret.mean() 

        return ret

## 1.3 Adversarial Attack

In [7]:
# from torch_scatter import scatter_add
from torch_geometric.utils import add_self_loops
from torch import Tensor
from typing import Tuple

def normalize_adj_tensor(adj):
    """Symmetrically normalize adjacency tensor."""
    rowsum = torch.sum(adj,1)
    d_inv_sqrt = torch.pow(rowsum, -0.5)
    d_inv_sqrt[d_inv_sqrt == float("Inf")] = 0.
    d_mat_inv_sqrt = torch.diag(d_inv_sqrt)
    return torch.mm(torch.mm(adj,d_mat_inv_sqrt).transpose(0,1),d_mat_inv_sqrt)

def normalize_adj_tensor_sp(adj):
    """Symmetrically normalize sparse adjacency tensor."""
    device = adj.device
    adj = adj.to("cpu")
    rowsum = torch.spmm(adj, torch.ones((adj.size(0),1))).reshape(-1)
    d_inv_sqrt = torch.pow(rowsum, -0.5)
    d_inv_sqrt[d_inv_sqrt == float("Inf")] = 0.
    d_mat_inv_sqrt = torch.diag(d_inv_sqrt)
    adj = torch.mm(torch.smm(adj.transpose(0,1),d_mat_inv_sqrt.transpose(0,1)),d_mat_inv_sqrt)
    return adj.to(device)

def edge2adj(x, edge_index):
    """Convert edge index to adjacency matrix"""
    num_nodes = x.shape[0]
    tmp, _ = add_self_loops(edge_index, num_nodes=num_nodes)
    edge_weight = torch.ones(tmp.size(1), dtype=None,
                                     device=edge_index.device)
    row, col = tmp[0], tmp[1]
    # deg = scatter_add(edge_weight, row, dim=0, dim_size=num_nodes)
    deg = torch.zeros(num_nodes, dtype=edge_weight.dtype, device=edge_weight.device)
    deg.scatter_add_(0, row, edge_weight)
    
    deg_inv_sqrt = deg.pow_(-0.5)
    deg_inv_sqrt.masked_fill_(deg_inv_sqrt == float('inf'), 0)
    edge_weight = deg_inv_sqrt[row] * edge_weight * deg_inv_sqrt[col]
    #return torch.sparse.FloatTensor(tmp, edge_weight,torch.Size((num_nodes, num_nodes)))
    return torch.sparse_coo_tensor(tmp, edge_weight, torch.Size((num_nodes, num_nodes)))

def EdgeDropping(edge_index: Tensor, p: float = 0.5,
                 force_undirected: bool = False,
                 training: bool = True) -> Tuple[Tensor, Tensor]:
    if p < 0. or p > 1.:
        raise ValueError(f'Dropout probability has to be between 0 and 1 '
                         f'(got {p}')

    if not training or p == 0.0:
        edge_mask = edge_index.new_ones(edge_index.size(1), dtype=torch.bool)
        return edge_index, edge_mask

    row, col = edge_index

    edge_mask = torch.bernoulli(torch.ones(row.size(0), device=edge_index.device) * (1 - p)).type(torch.bool)

    if force_undirected:
        edge_mask[row > col] = False

    edge_index = edge_index[:, edge_mask]

    if force_undirected:
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_mask = edge_mask.nonzero().repeat((2, 1)).squeeze()

    return edge_index, edge_mask


def GeneDropping(x, drop_prob):
    drop_mask = torch.bernoulli(torch.ones(x.size(1), device=x.device) * (1 - drop_prob)).type(torch.bool) 
    x = x.clone()
    x[:, drop_mask] = 0
    return x

def GraphAdversarialAttack(model, edge_index_1, edge_index_2, x_1, x_2, iters, node_ratio, alpha, beta):
    """ PGD attack on both features and edges"""

    for param in  model.parameters():
        param.requires_grad = False
    
    model.eval()
    device = x_1.device
    total_edges = edge_index_2.shape[1]         
    n_node = x_2.shape[0]
    eps = total_edges * node_ratio/2
    xi = 1e-3
    
    A_ = torch.sparse_coo_tensor(edge_index_2, torch.ones(total_edges,device=device), torch.Size((n_node, n_node))).to_dense() 
    C_ = torch.ones_like(A_) - 2 * A_ - torch.eye(A_.shape[0],device=device)
    S_ = torch.zeros_like(A_, requires_grad= True)
    mask = torch.ones_like(A_)
    mask = mask - torch.tril(mask)
    delta = torch.zeros_like(x_2, device=device, requires_grad=True)
    adj_1 = edge2adj(x_1, edge_index_1)
    model.to(device)

    for itr in range(iters):
        S = (S_ * mask)
        S = S + S.T
        A_prime = A_ + (S * C_)
        adj_hat = normalize_adj_tensor(A_prime + torch.eye(n_node,device=device))

        z1 = model(x_1, adj_1)
        z2 = model(x_2 + delta, adj_hat) 
        Attackloss = model.loss(z1, z2, batch_size=0) 
        Attackloss.backward()

        S_.data = (S_.data + alpha/np.sqrt(itr+1)*S_.grad.detach()) # annealing
        S_.data = bisection(S_.data, eps, xi) # clip S
        S_.grad.zero_()
        
        delta.data = (delta.data + beta*delta.grad.detach().sign()).clamp(-0.04,0.04)        
        delta.grad.zero_()

        randm = torch.rand(n_node, n_node,device=device)
        discretized_S = torch.where(S_.detach() > randm, torch.ones(n_node, n_node,device=device), torch.zeros(n_node, n_node, device=device))
        discretized_S = discretized_S + discretized_S.T
        A_hat = A_ + discretized_S * C_ + torch.eye(n_node,device=device)
        
    for param in model.parameters():
        param.requires_grad = True
    
    model.train()
    x_hat = x_2 + delta.data.to(device)
    assert torch.equal(A_hat, A_hat.transpose(0,1))
    return normalize_adj_tensor(A_hat), x_hat

def bisection(a,eps,xi,ub=1):
    pa = torch.clamp(a, 0, ub)
    if torch.sum(pa) <= eps:
        upper_S_update = pa
    else:
        mu_l = torch.min(a-1)
        mu_u = torch.max(a)
        mu_a = (mu_u + mu_l)/2
        while torch.abs(mu_u - mu_l)>xi:
            mu_a = (mu_u + mu_l)/2
            gu = torch.sum(torch.clamp(a-mu_a, 0, ub)) - eps
            gu_l = torch.sum(torch.clamp(a-mu_l, 0, ub)) - eps
            if gu == 0:
                break
            if torch.sign(gu) == torch.sign(gu_l):
                mu_l = mu_a
            else:
                mu_u = mu_a
        upper_S_update = torch.clamp(a-mu_a, 0, ub)
    return upper_S_update

## 1.4 Topology Signature loss function

In [8]:
from typing import Optional, Callable
from aleph import vietoris_rips_from_matrix
from math import prod

class TracebackIndexError(Exception):
    """
    Traceback index error when edge distance not found in distance matrix.
    """
    def __init__(self, edge_distance: float, distance_matrix: np.ndarray) -> None:
        self.edge_distance = edge_distance
        self.distance_matrix = distance_matrix
        super().__init__(f"Edge distance {edge_distance} not found in distance matrix.")

def get_euclidean_distance_matrix(tensors: torch.Tensor) -> torch.Tensor:
    """
        Return euclidean distance matrix between all data points
    """
    diff = tensors.unsqueeze(1) - tensors.unsqueeze(0)  # shape (n, n, d)
    d2 = (diff * diff).sum(-1)
    dist = torch.sqrt(d2 + 1e-12)  # add small epsilon for stability
    
    return dist

class OptimalTopologySignatureLoss(nn.Module):
    """
    New update:
        Cache permutation indices for batching to ensure consistent batching between X and Z. Only update the permutation after some epochs.
        Cache persistence pairs for each batch and cache the persistence homology indices for X_tensor to avoid recomputing them every time.

    """
    max_dimension: int
    batch_size: Optional[int]
    shuffle_batch: bool
    X_tensor: torch.Tensor              # Original data cloud that we want to be the mock
    traceback_error_skip: bool
    dist_matrix_fn: Callable[[torch.Tensor], np.ndarray]
    turnoff_traceback_warning: bool

    def __init__(self, X_tensor: torch.Tensor,
                 max_dimension: int = 2, batch_size: Optional[int] = None, 
                 shuffle_batch: bool = True, traceback_error_skip: bool = True, 
                 turnoff_traceback_warning: bool = True,
                 distance_fn: Optional[Callable[[torch.Tensor, torch.Tensor], float]] = None):
        
        super(OptimalTopologySignatureLoss, self).__init__()
        self.max_dimension = max_dimension
        self.traceback_error_skip = traceback_error_skip
        self.batch_size = batch_size
        self.shuffle_batch = shuffle_batch
        self.turnoff_traceback_warning = turnoff_traceback_warning

        # Construct distance matrix function
        if distance_fn is not None:
            self.dist_matrix_fn = self._get_distance_fn_matrix(distance_fn)
        else:
            self.dist_matrix_fn = get_euclidean_distance_matrix
        
        # Cache things for optimize training
        self.data_size = X_tensor.size(0)
        self.X_tensor = X_tensor
        self.cached_yX = None

    def _sampling_batch_indices(self):
        """
        Get random permutation indices for batching. And generate the 
        """
        self.permutation_indices = torch.randperm(self.data_size)
        self.cached_yX = []

        # Get batches and load significant indexes into cache
        perumutation_X = self.X_tensor[self.permutation_indices]
        num_batches = (self.data_size + self.batch_size - 1) // self.batch_size
        for i in range(num_batches):
            start_idx = i * self.batch_size
            end_idx = min((i + 1) * self.batch_size, self.data_size)
            X_batch = perumutation_X[start_idx:end_idx]

            # Cache the indices pairs for this batch
            self.cached_yX.append(self._get_indices_pairs(self.dist_matrix_fn(X_batch)))

    def _get_distance_fn_matrix(self, dist_fn: Callable[[torch.Tensor, torch.Tensor], float]) -> Callable[[torch.Tensor], np.ndarray]:
        """Get the distance matrix function.

        Args:
            dist_fn: distance function to use

        Returns:
            - Distance matrix function
        """
        def dist_matrix_fn(tensor: torch.Tensor) -> np.ndarray:
            matrix = np.zeros((len(tensor), len(tensor)))
            for i in range(len(tensor)):
                for j in range(i + 1, len(tensor)):
                    dist = dist_fn(tensor[i], tensor[j])
                    matrix[i, j] = dist
                    matrix[j, i] = dist
            return matrix

        return dist_matrix_fn

    def _traceback_index(self, edge_distance: float, distance_matrix: np.ndarray) -> tuple[int, int]:
        """Given an edge distance, find the corresponding edge (i, j) in the distance matrix. Find the edge that created the feature by looking at the distance matrix.

        Args:
            edge_distance: edge distance
            distance_matrix: numpy array of distances

        Returns:
            - Edge (i, j) as tuple of indices
        """
        indices = np.where(distance_matrix.detach().cpu() == edge_distance)
        if len(indices[0]) == 0:
            if not self.traceback_error_skip:
                return -1, -1
            else:
                raise TracebackIndexError(edge_distance, distance_matrix)
        indices = (indices[0].tolist(), indices[1].tolist())
        if len(indices[0]) > 2 and not self.turnoff_traceback_warning:                 # Remove permutation of same indices pairs:
            print(f"Warning: multiple edges found with the same distance: {list(zip(indices[0], indices[1]))}. Returning the first one.")
        return indices[0][0], indices[1][0]
    
    def _get_indices_pairs(self, distance_matrix: np.ndarray) -> list[tuple[int, int]]:
        """Given a tensor of shape (N, d), compute the distance matrix and return the edge pairs that create the Homology features (from H0 to H_dimension).

        Args:
            tensor: numpy array of shape (N, d)
        Returns:
            - List of edge pairs (i, j) as tuples of indices
        """
        pairs = vietoris_rips_from_matrix(distance_matrix.detach().cpu().numpy(), max_dimension=self.max_dimension)

        # Get edge pairs from H0 first. For H0, the birth is always 0, so we only care about the death
        edge_pairs = [self._traceback_index(point.y, distance_matrix) for point in pairs[0] if point.y != float('inf')]
        
        # Then get edge pairs from H1 to H_dimension
        for dim in range(1, len(pairs)):
            pairs[dim].removeDiagonal()  # Remove diagonal points (0 persistence)
            edge_pairs.extend(self._traceback_index(point.x, distance_matrix) for point in pairs[dim])
            edge_pairs.extend(self._traceback_index(point.y, distance_matrix) for point in pairs[dim])
            
        edge_pairs = [pair for pair in edge_pairs if pair != (-1, -1)]  # Remove invalid pairs
        return edge_pairs
        
    def forward(self, Z_tensor: torch.Tensor, reshuffle: bool = True) -> torch.Tensor:

        if self.batch_size is not None:
            total_loss = 0
            num_batches = (Z_tensor.shape[0] + self.batch_size - 1) // self.batch_size

            # Shuffle the data
            if reshuffle or self.cached_yX is None:
                self._sampling_batch_indices()

            Z_tensor = Z_tensor[self.permutation_indices]
            X_tensor = self.X_tensor[self.permutation_indices]

            for i in range(num_batches):
                start_idx = i * self.batch_size
                end_idx = min((i + 1) * self.batch_size, Z_tensor.shape[0])

                Z_batch = Z_tensor[start_idx:end_idx]
                X_batch = X_tensor[start_idx:end_idx]

                total_loss += self._forward(Z_batch, X_batch, batch_index=i)
            return total_loss / num_batches
        else:
            return self._forward(Z_tensor)

    def _forward(self, Z_tensor: np.ndarray, X_tensor: np.ndarray, batch_index: int) -> torch.Tensor:
        # Placeholder for actual topology signature loss computation
        """
            Compute perstence loss between two tensors.

            New update: added caching for the indices of the X_tensor data cloud.
        """
        # Normalize for both to make sure the consistency loss
        Z_norm = nn.functional.normalize(Z_tensor, dim=0) / prod(s for s in Z_tensor.shape[1:])
        X_norm = nn.functional.normalize(X_tensor, dim=0) / prod(s for s in X_tensor.shape[1:])

        mZ = self.dist_matrix_fn(Z_norm)
        mX = self.dist_matrix_fn(X_norm)

        # Index of data cloud before encoding and after encoding
        y_X = self.cached_yX[batch_index]
        y_Z = self._get_indices_pairs(mZ)

        # Compute distances for the minimum spanning trees
        mX_yX = [mX[x, y] for x, y in y_X]
        mX_yZ = [mX[x, y] for x, y in y_Z]

        mZ_yX = [mZ[x, y] for x, y in y_X]
        mZ_yZ = [mZ[x, y] for x, y in y_Z]

        loss = 1/2 * sum((x - y)**2 for x, y in zip(mX_yX, mX_yZ)) + sum((x - y)**2 for x, y in zip(mZ_yZ, mZ_yX))
        
        # Normalized by batch size and multiply by the product of the shape of Z_tensor except the batch size
        return loss / X_tensor.shape[0] * prod(s for s in Z_tensor.shape[1:])


## 1.5 Train function

In [10]:
from time import perf_counter as t
from sklearn.decomposition import PCA

def Train(scAGCLmodel, data, cellGgraph, device, num_epochs, lam, alpha, beta, iters, optimizer, edge_r1, edge_r2, feature_r1, feature_r2, subgraph_size, topo_loss_weight=0):
    start_time = t()
    scAGCLmodel.train()   
   
    Z_final=None
    bessloss=None
    
    for epoch in range(1, num_epochs + 2): #do not use final round
        # 1. Compute the Adversarial mode first
        scAGCLmodel.eval()
        X=data.x
        Adj = edge2adj(X, data.edge_index)
        X = X.to(device)
        Adj = Adj.to(device)
        Z_current = scAGCLmodel(X, Adj) #get embeddings of the original input

        # Stage 2: attack the graph and use the augmented graph to train
        scAGCLmodel.train() #back to continue training

        subGraph = cellGgraph.subgraph(np.random.permutation(cellGgraph.number_of_nodes())[:subgraph_size])
        x_sub = data.x[np.array(subGraph.nodes())].to(device)
        subGraph = nx.relabel.convert_node_labels_to_integers(subGraph, first_label=0, ordering='default')      
        edgeind = np.array(subGraph.edges()).T
        edgeind = torch.LongTensor(np.hstack([edgeind,edgeind[::-1]])).to(device)

        optimizer.zero_grad()
        x_1 = GeneDropping(x_sub, feature_r1)
        x_2 = GeneDropping(x_sub, feature_r2)

        edgeind_1 = EdgeDropping(edgeind, p=edge_r1, force_undirected=True)[0]
        edgeind_2 = EdgeDropping(edgeind, p=edge_r2, force_undirected=True)[0]
        adj_1 = edge2adj(x_1, edgeind_1)
        adj_2 = edge2adj(x_2, edgeind_2)

        z_1 = scAGCLmodel(x_1, adj_1)
        z_2 = scAGCLmodel(x_2, adj_2)    
        loss1= scAGCLmodel.loss(z_1,z_2,batch_size=0)
        
        loss2=0
        if lam > 0:
            adj_3, x_3 = GraphAdversarialAttack(scAGCLmodel, edgeind, edgeind_1, x_sub, x_1, iters, 0.2, alpha, beta)
            adj_4, x_4 = GraphAdversarialAttack(scAGCLmodel, edgeind, edgeind_2, x_sub, x_2, iters, 0.2, alpha, beta)
            z_3 = scAGCLmodel(x_3,adj_3)
            z_4 = scAGCLmodel(x_4,adj_4)
            loss2 = scAGCLmodel.loss(z_3,z_4,batch_size=0)

        # Including loss of topology ==> Align z_1 with z_2 and z_3 with z_4
        loss3 = 0
        # if topo_loss_weight > 0:
        #     x_compressed = PCA(n_components=100).fit_transform(x_sub.cpu())
        #     x_compressed = torch.tensor(x_compressed).to(device)
        #     topology_loss_fn = OptimalTopologySignatureLoss(X_tensor=x_compressed, max_dimension=2, batch_size=64)
        #     loss3 = topo_loss_weight * (topology_loss_fn.forward(z_1, reshuffle=False) 
        #                                 + topology_loss_fn.forward(z_2, reshuffle=False))
        
        if topo_loss_weight > 0:
            x_compressed = PCA(n_components=100).fit_transform(x_sub.cpu())
            x_compressed = torch.tensor(x_compressed).to(device)
            topology_loss_fn = OptimalTopologySignatureLoss(X_tensor=x_compressed, max_dimension=2, batch_size=64)
            
            loss_topo_z1 = topology_loss_fn.forward(z_1, reshuffle=False)
            loss_topo_z2 = topology_loss_fn.forward(z_2, reshuffle=False)
            
            loss_topo_z3 = topology_loss_fn.forward(z_3, reshuffle=False)
            loss_topo_z4 = topology_loss_fn.forward(z_4, reshuffle=False)
            
            loss3 = topo_loss_weight * (loss_topo_z1 + loss_topo_z2 + loss_topo_z3 + loss_topo_z4)
        
        loss = loss1 + lam*loss2 + topo_loss_weight*loss3

        loss.backward()
        optimizer.step()

        if epoch <= num_epochs and epoch%20==0:
            now_time = t()
            print(f'Epoch={epoch:03d}, loss1={loss1:.4f}, loss2={loss2:.4f}, loss3={loss3:.4f}, total loss={loss:.4f}, total time {now_time - start_time:.4f}')

        if epoch == 1:
            bessloss = loss
            Z_final=Z_current
            
        if loss < bessloss:
            bessloss = loss
            Z_final=Z_current

    return Z_final

# 2. Training Model

In [12]:
RANDOM_SEED = 12345

# Config for model
subgraph_size = 256
num_proj_hidden = 256
num_hidden = 256
num_layers = 2
tau = 0.5
learning_rate = 0.001
weight_decay = 1e-5

num_epochs = 500
num_itersAdv = 10

lam = 1
alpha = 100
beta = 0.01
edge_r1 = 0.4
edge_r2 = 0.3
feature_r1 = 0.3
feature_r2 = 0.4

## 2.1 Training scAGCL Model

In [13]:
import random
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import normalize as sklearn_normalize

torch.manual_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATASET = "QS_Diaphragm"
# DATASET = "yan"
DATA_PATH = f"/kaggle/input/datasets/quocchieu586/scrna-h5/{DATASET}.h5"

X, Y = load_h5_data1(DATA_PATH)
NUM_CLUSTERS = len(set(Y))

# Read input data
data = Data(x=torch.from_numpy(X))
cellGraph=GraphConstruction(data, X, NUM_CLUSTERS)
data.y=torch.tensor(Y, dtype=torch.int64)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(data.num_features, num_hidden, num_layers=num_layers).to(device)
model = AGCLModel(encoder, num_hidden, num_proj_hidden, tau).to(device)
#optimizer   
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

embeddings_1=Train(model, data, cellGraph, device, num_epochs,lam, alpha, beta, num_itersAdv, optimizer, 
                 edge_r1, edge_r2, feature_r1, feature_r2, subgraph_size, topo_loss_weight=0)

Z_eval = embeddings_1.clone()
Y=data.y
Z = Z_eval.detach().cpu().numpy()
Y = Y.detach().cpu().numpy()
Z = sklearn_normalize(Z, norm='l2')

kmeans = KMeans(n_clusters=NUM_CLUSTERS, init="k-means++", random_state=0)
pred = kmeans.fit_predict(Z)

ari_score=adjusted_rand_score(Y, pred)
nmi_score=normalized_mutual_info_score(Y, pred)

print('Final result without topology loss: ARI= ' + str(ari_score) + ', NMI=' + str(nmi_score)) 

Reading data!
There are 870 cells, 23433 genes
Data Preprocessing!
Keeping 2000 genes
Building a Graph with 870 nodes and 10598 edges
Epoch=020, loss1=5.2744, loss2=6.4798, loss3=0.0000, total loss=11.7542, total time 5.0343
Epoch=040, loss1=5.2335, loss2=5.6442, loss3=0.0000, total loss=10.8777, total time 9.0323
Epoch=060, loss1=5.2499, loss2=5.8390, loss3=0.0000, total loss=11.0889, total time 12.9702
Epoch=080, loss1=5.2740, loss2=5.6268, loss3=0.0000, total loss=10.9008, total time 16.9818
Epoch=100, loss1=5.2856, loss2=5.6590, loss3=0.0000, total loss=10.9446, total time 20.9839
Epoch=120, loss1=5.2858, loss2=5.5938, loss3=0.0000, total loss=10.8795, total time 25.0209
Epoch=140, loss1=5.1785, loss2=5.8367, loss3=0.0000, total loss=11.0153, total time 29.0282
Epoch=160, loss1=5.2213, loss2=5.6642, loss3=0.0000, total loss=10.8856, total time 33.0024
Epoch=180, loss1=5.1683, loss2=5.4896, loss3=0.0000, total loss=10.6578, total time 36.9810
Epoch=200, loss1=5.2458, loss2=5.6452, l

In [14]:
encoder = Encoder(data.num_features, num_hidden, num_layers=num_layers).to(device)
model = AGCLModel(encoder, num_hidden, num_proj_hidden, tau).to(device)
#optimizer   
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

embeddings_2=Train(model, data, cellGraph, device, num_epochs,lam, alpha, beta, num_itersAdv, optimizer, 
                 edge_r1, edge_r2, feature_r1, feature_r2, subgraph_size, topo_loss_weight=10)

Z_eval = embeddings_2.clone()
Y=data.y
Z = Z_eval.detach().cpu().numpy()
Y = Y.detach().cpu().numpy()
Z = sklearn_normalize(Z, norm='l2')

kmeans = KMeans(n_clusters=NUM_CLUSTERS, init="k-means++", random_state=0)
pred = kmeans.fit_predict(Z)

ari_score=adjusted_rand_score(Y, pred)
nmi_score=normalized_mutual_info_score(Y, pred)

print('Final result with topology loss: ARI= ' + str(ari_score) + ', NMI=' + str(nmi_score)) 

Epoch=020, loss1=5.5053, loss2=6.1814, loss3=0.0880, total loss=12.5671, total time 106.3675
Epoch=040, loss1=5.2742, loss2=5.8099, loss3=0.0957, total loss=12.0405, total time 216.5403
Epoch=060, loss1=5.2546, loss2=5.8934, loss3=0.0875, total loss=12.0225, total time 329.5946
Epoch=080, loss1=5.2421, loss2=5.5913, loss3=0.0795, total loss=11.6286, total time 442.3527
Epoch=100, loss1=5.2400, loss2=5.9148, loss3=0.0774, total loss=11.9290, total time 558.0826
Epoch=120, loss1=5.2761, loss2=5.7647, loss3=0.0887, total loss=11.9276, total time 674.6928
Epoch=140, loss1=5.2693, loss2=5.8525, loss3=0.0911, total loss=12.0324, total time 792.7133
Epoch=160, loss1=5.2589, loss2=5.6296, loss3=0.0832, total loss=11.7206, total time 922.3208
Epoch=180, loss1=5.2523, loss2=5.9028, loss3=0.0910, total loss=12.0651, total time 1042.1888
Epoch=200, loss1=5.2083, loss2=5.5692, loss3=0.0841, total loss=11.6180, total time 1152.3701
Epoch=220, loss1=5.2432, loss2=5.5963, loss3=0.0945, total loss=11.7